In [1]:
import os
import sys

import pandas as pd
import threading
import time
from IPython.display import display
from rich.jupyter import JupyterMixin
from rich.table import Table
from rich.panel import Panel
from rich.layout import Layout
import importlib
import shutil
import re

# Import trading library
from tradeSim import TradeSim
from tradeSim import StrategyHandler

# **__Trading simulation__**

Welcome to the simulation platform!

This Python-based environment is designed to replicate a realistic trading experience using the SET50 daily ticks dataset. It supports essential features like real-time data streaming, order matching, portfolio management, and fee calculations, giving you everything you need to build and test your algorithmic trading strategies.

Each participant begins with a virtual portfolio containing 10,000,000 Baht in starting capital. Your challenge is to design effective trading algorithms and compete with other teams to generate the highest return. Think smart, trade fast, and enjoy the competition!

---

### **Import Daily ticks information for trading infomation**
For each day within compitition, the tick information will be provided daily

In [2]:
#TODO: Change ticks information daily
daily_ticks = "./marketInfo/ticks/2026-09-09.csv"

df = pd.read_csv(daily_ticks)
df['TradeDateTime'] = pd.to_datetime(df['TradeDateTime'])
# แยกข้อมูลตาม symbol ครั้งเดียวตรงนี้ แล้วเก็บเป็น list ของ dict
# (การวน groupby ซ้ำทุก tick ทำให้ pandas สร้าง group ใหม่ทั้งก้อนทุกรอบ
#  และ .iloc[i] สร้าง Series ใหม่ทุกแถว -- ทั้งคู่อยู่ใน loop ที่วนหลักแสนรอบ
#  ต่อหนึ่งวัน จึงกินเวลาเกือบทั้งหมดของการรัน)
ticks_by_symbol = {
    symbol: group.to_dict("records")
    for symbol, group in df.groupby('ShareCode')
}

### **Now, name your team!**

The code section below will create your trading simulation class using the team name you provide.

- Your team name will be set as the `owner` of the `TraderPortfolio` from `trading_sim`.
- The `SimExecution` class will handle order execution for buying stocks in the market.

**Team name requirements:**
- Must use English letters or numbers  
- No spaces allowed  
- Hyphens (`-`) and underscores (`_`) are allowed  
- Must be **30 characters or fewer**

**`fresh_start`:**
- `True` (default): every run of this cell starts from a fresh 10,000,000 portfolio -- use this while trying out strategies
- `False`: carries on from the portfolio saved in `result/<team_name>/` by the last run -- use this when practising one day at a time (change the date above, then run again)

In [3]:
#TODO: Replace with your team name
team_name = "Full_option"
strategy_name = "ConservativeAlphaHunter"
# True  = start from a fresh 10,000,000 every time this cell runs (deletes result/<team_name>/)
# False = carry on from the saved portfolio -- use this when practising one day at a time
fresh_start = True

pattern = r'^[A-Za-z0-9-_]{1,30}$'
if not bool(re.match(pattern, team_name)) or not bool(re.match(pattern, strategy_name)):
    raise ValueError("Team name or strategy name is invalid. Please use only alphanumeric characters, hyphens, and underscores, with a maximum length of 30 characters.")

# Must happen before tradeSim() below: that is where the saved portfolio is loaded
if fresh_start:
    shutil.rmtree(os.path.join("result", team_name), ignore_errors=True)

# Init trade system
# current_date is the day being simulated (from the tick file loaded above) --
# it bounds get_historical_data() so a strategy can never see this day's own
# data, even when testing locally.
trading_Sim = TradeSim.tradeSim(team_name, current_date=df['TradeDateTime'].dt.date.iloc[0])
strategy_runner = trading_Sim.get_strategy_runner()

[INFO] Created new portfolio for 'Full_option'


### **Craft Your Strategy**

To simulate your trading strategy within the platform,  
you must create your own strategy class by inheriting from the `strategy_template` class.

Once you've finished writing or are ready to test it,  
run the code below to import your strategy into this notebook.


In [4]:
try:
    strategy_module = importlib.import_module(f"strategy.{strategy_name}")
    importlib.reload(strategy_module)
except ImportError as e:
    # Raise instead of printing and carrying on: the next line needs
    # strategy_module, so continuing here used to end in a confusing
    # "NameError: name 'strategy_module' is not defined" as the LAST line of
    # output, burying the real cause (usually just a typo in strategy_name).
    if (not os.path.exists(os.path.join("strategy", f"{strategy_name}.py"))
            and os.path.exists(os.path.join(os.pardir, "strategy", f"{strategy_name}.py"))):
        # Windows' "Extract All" offers a path ending in a new folder named
        # after the zip -- accept it and the notebook lands one level below
        # the repo, next to the zip's example strategy/ instead of the team's.
        raise ImportError(
            f"strategy/{strategy_name}.py is in the folder ABOVE this notebook "
            f"({os.path.abspath(os.pardir)}), not next to it: the zip was extracted into its own "
            f"subfolder ('{os.path.basename(os.getcwd())}'), which Windows' Extract All adds by "
            f"default. Move everything in this folder up into your repo folder (or extract the zip "
            f"again and delete that folder name from the path), then open TradingSimulation.ipynb "
            f"from the repo folder."
        ) from e
    raise ImportError(
        f"Could not import strategy/{strategy_name}.py -- check that the file exists "
        f"in the strategy/ folder and that strategy_name in the cell above matches its "
        f"file name EXACTLY (case-sensitive, no .py). Original error: {e}"
    ) from e

strategy_class = getattr(strategy_module, strategy_name, None)
if strategy_class is None:
    # Same rule the real submission check (CI) enforces -- catching it here
    # means finding out now instead of on a red Checks tab after pushing.
    raise AttributeError(
        f"strategy/{strategy_name}.py was imported, but it has no class named "
        f"'{strategy_name}'. The class name must match the file name exactly, because "
        f"the daily run looks the class up by file name."
    )

----

## **The Simulation**
The simulation models a real-time market environment using a thread handler, where each thread streams data for a specific symbol in the SET50 index, as provided in ```Daily_ticks.CSV```. This section of the code iterates through all entries in the CSV file, applying each competitor’s strategy based on the current market data. During the streaming process, the system attempts to match any pending orders in the order books and updates the market prices in the competitors’ portfolios for any held stocks.
##### *For more detailed explanation please look at document file*
>By setting `with_visual` to `False`, the simulation runs significantly faster.

In [5]:
#TODO: Change the `with_visual` according to your preference
# True if you want to see the visual simulation  
# False if you want to finish the simulation faster
with_visual = False

In [6]:
latest_prices = {}

# === Render Helpers ===
def render_market_table():
    table = Table(title="📈 Market Stream")
    table.add_column("Symbol")
    table.add_column("Last Price", justify="right")
    table.add_column("Volume", justify="right")
    table.add_column("Flag", justify="right")

    snapshot = dict(latest_prices)  # copy to avoid mid-update issues

    for sym, data in snapshot.items():
        table.add_row(
            sym,
            f"{data['price']:.2f}",
            str(data["volume"]),
            data["Flag"]
        )
    return table


def render_portfolio_table():
    table = Table(title=f"💼 Portfolio: {strategy_runner.get_owner()}", expand=True)
    table.add_column("Cash Balance", no_wrap=True, width=18)
    table.add_column("Symbol")
    table.add_column("Actual Vol", justify="right")
    table.add_column("Buy Price", justify="right")
    table.add_column("Market Price", justify="right")
    table.add_column("Average Cost", justify="right")
    table.add_column("Amount Cost", justify="right")
    table.add_column("Market Value", justify="right")
    table.add_column("Unrealized", justify="right")
    table.add_column("Unreal. %", justify="right")
    table.add_column("Realized", justify="right")
    table.add_column("Buy Time", justify="center")

    stock_infos = strategy_runner.get_all_stocks_info()
    cash_str = f"{strategy_runner.get_cash_balance():,.2f}"
    cash_displayed = False

    if not stock_infos:
        table.add_row(cash_str, "-", "0", "-", "-", "-", "-", "-", "-", "-", "-", "-")
    else:
        for stock in stock_infos:
            table.add_row(
                cash_str if not cash_displayed else "",
                stock["Symbol"],
                str(stock["Actual Volume"]),
                f"{stock['Buy Price']:.2f}",
                f"{stock['Market Price']:.2f}",
                f"{stock['Average Cost']:.2f}",
                f"{stock['Amount Cost']:.2f}",
                f"{stock['Market Value']:.2f}",
                f"{stock['Unrealized P&L']:.2f}",
                f"{stock['Unrealized %']:.2f}%",
                f"{stock['Realized P&L']:.2f}",
                stock["Buy time"]
            )
            cash_displayed = True

    # Add metrics summary
    table.add_section()
    table.add_row("Metrics", "", "", "", "", "", "", "", "", "", "", "")
    # One get_portfolio_info() call for every row: each metric walks the whole
    # Round-Trip list, and this table is redrawn throughout the run.
    # Benchmark/Excess are None until save_summary_csv() at the end of the day
    info = strategy_runner.get_portfolio_info()
    pct = lambda v: "-" if v is None else f"{v:.2f}%"
    table.add_row("Return (%)", f"{info['Return rate']:.2f}%", "", "", "", "", "", "", "", "", "", "")
    table.add_row("Benchmark Return (%)", pct(info["Benchmark Return"]), "", "", "", "", "", "", "", "", "", "")
    table.add_row("Excess Return (%)", pct(info["Excess Return"]), "", "", "", "", "", "", "", "", "", "")
    table.add_row("Max Drawdown (%)", f"{info['Max Drawdown (%)']:.2f}%", "", "", "", "", "", "", "", "", "", "")
    table.add_row("Buy Orders", str(info["Buy Order Count"]), "", "", "", "", "", "", "", "", "", "")
    table.add_row("Sell Orders", str(info["Sell Order Count"]), "", "", "", "", "", "", "", "", "", "")
    table.add_row("Round-Trips", str(info["Round-Trip Count"]), "", "", "", "", "", "", "", "", "", "")
    table.add_row("Profitable Round-Trips", str(info["Profitable Round-Trips"]), "", "", "", "", "", "", "", "", "", "")
    table.add_row("Win Rate", f"{info['Win Rate']:.2f}%", "", "", "", "", "", "", "", "", "", "")
    table.add_row("Expected Payoff", f"{info['Expected Payoff']:.2f}", "", "", "", "", "", "", "", "", "", "")
    table.add_row("Avg Gain", f"{info['Avg Gain']:.2f}", "", "", "", "", "", "", "", "", "", "")
    table.add_row("Avg Loss", f"{info['Avg Loss']:.2f}", "", "", "", "", "", "", "", "", "", "")
    table.add_row("Avg Exposure", f"{info['Average Exposure'] * 100:.2f}%", "", "", "", "", "", "", "", "", "", "")
    return table


def render_layout():
    layout = Layout()
    layout.split_column(
        Layout(Panel(render_market_table(), expand=True), name="market"),
        Layout(Panel(render_portfolio_table(), expand=True), name="portfolio")
    )
    return layout


# rich's Live draws through an ipywidgets Output widget inside Jupyter, and
# VS Code's renderer can fail to load it ("No version of module
# @jupyter-widgets/output is registered"). NotebookLive refreshes a plain
# IPython display instead, so the live view needs no widgets at all.
class _RichView(JupyterMixin):
    def __init__(self, renderable):
        self.renderable = renderable

    def __rich__(self):
        return self.renderable


class NotebookLive:
    def __init__(self, renderable, refresh_per_second=2):
        self.view = _RichView(renderable)
        self.interval = 1 / refresh_per_second
        self._stop = threading.Event()

    def _run(self):
        while not self._stop.wait(self.interval):
            self.handle.update(self.view)

    def __enter__(self):
        self.handle = display(self.view, display_id=True)
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()
        self.handle.update(self.view)


# === Simulation Core ===
# strategy_dir is what lets handler.load_json()/load_model()/get_companion_path()
# find a config or trained model submitted next to the strategy file -- without
# it those calls raise, so a strategy that works on the VPS would fail here.
strategy_dir = os.path.dirname(strategy_module.__file__)

handlers = {
    symbol: StrategyHandler.StrategyHandler(strategy_class, strategy_runner, strategy_dir)
    for symbol in ticks_by_symbol
}

max_length = max(len(rows) for rows in ticks_by_symbol.values())
layout = render_layout()
live = None

if with_visual:
    # Rebuild the tables no more often than the screen refreshes -- rebuilding
    # them after every tick only to have NotebookLive show one of them every
    # half second was most of visual mode's running time.
    refresh_per_second = 2
    next_render = 0.0
    with NotebookLive(layout, refresh_per_second=refresh_per_second) as live:
        for tick in range(max_length):
            for symbol, rows in ticks_by_symbol.items():
                if tick >= len(rows):
                    continue

                row = rows[tick]
                price_update = {row['ShareCode']: row['LastPrice']}

                handler = handlers[symbol]
                handler.process_row(row)

                if not trading_Sim.isOrderbooksEmpty():
                    trading_Sim.isMatch(row)

                trading_Sim.update_market_prices(price_update)

                latest_prices[symbol] = {
                    "price": row['LastPrice'],
                    "volume": row['Volume'],
                    "Flag": row['Flag']
                }

            if time.monotonic() >= next_render:
                layout["market"].update(Panel(render_market_table()))
                layout["portfolio"].update(Panel(render_portfolio_table()))
                next_render = time.monotonic() + 1 / refresh_per_second

else:
    # Headless (no visual mode)
    for tick in range(max_length):
        for symbol, rows in ticks_by_symbol.items():
            if tick >= len(rows):
                continue

            row = rows[tick]
            price_update = {row['ShareCode']: row['LastPrice']}

            handler = handlers[symbol]
            handler.process_row(row)

            if not trading_Sim.isOrderbooksEmpty():
                trading_Sim.isMatch(row)

            trading_Sim.update_market_prices(price_update)

            latest_prices[symbol] = {
                "price": row['LastPrice'],
                "volume": row['Volume'],
                "Flag": row['Flag']
            }

# === Final flush/save ===
trading_Sim.flushTransactionLog()
trading_Sim.flushErrorLogger()
trading_Sim.create_transaction_summarize(team_name)

# Order matters, and matches the real daily run (main.py): save_summary_csv()
# internally calls finalize_day(), the only thing that appends to
# daily_snapshots and advances the running peak/trough behind Portfolio Max DD.
# Saving the portfolio BEFORE that would write today's portfolio.json without
# today's snapshot, so Max DD and Average Portfolio Exposure would silently
# reset to a single day's worth every time you run another day.
# (The real run also writes a holdings.csv here, via scripts/_common.py -- that
# folder is organizer-only, so locally just read cell 13's table instead.)
trading_date = df['TradeDateTime'].dt.date.iloc[0]
# Benchmark Return / Max DD have to be measured over the SAME window the
# portfolio has actually been running, or Excess Return subtracts two
# different periods from each other and the number means nothing. Without a
# start date the benchmark falls back to the earliest file in
# marketInfo/benchmark/, which is whatever day the bundled data happens to
# begin -- measured on the shipped data that reported the market up 6.33%
# over a week it actually fell 1.23%.
#
# daily_snapshots holds one entry per day already finalized, so its first
# entry is the day this portfolio started. On the very first run it is still
# empty (today's entry is appended inside save_summary_csv, below), and the
# right answer then is today.
_snapshots = trading_Sim.portfolio.daily_snapshots
benchmark_start = _snapshots[0]["date"] if _snapshots else str(trading_date)
trading_Sim.save_summary_csv(trading_date, university_start_date=benchmark_start)
trading_Sim.save_portfolio()

# Final redraw AFTER save_summary_csv(): that is where Benchmark/Excess
# Return become available (they stay "-" while the day is running)
layout["market"].update(Panel(render_market_table()))
layout["portfolio"].update(Panel(render_portfolio_table()))
if live is not None:
    live.handle.update(live.view)

[OK] Summary saved to c:\Users\porsche\Documents\tradeSim0.2.5\result\Full_option\Full_option_portfolios_transaction_summary.csv


### **Examine Portfolio Information**

You can view portfolio statistics using the provided methods.

For example:
```
strategy_runner.get_all_stock_info() -> Returns a summary of each stock in the portfolio.
strategy_runner.get_portfolio_info() -> Returns a summary of portfolio details such as cash balance, NAV, and ROI.
```
Full documentation is available in the provided PDF.

In [7]:
portInfo = strategy_runner.get_portfolio_info()
stockInfo = strategy_runner.get_all_stocks_info()
pd.DataFrame([portInfo]).T

,0
Owner,Full_option
Number of Stocks,0
Total Cost,0.0
Unrealized P&L,0.0
Unrealized %,0.0
Realized P&L,0.0
Cash Balance,10000000.0
Net Asset Value,10000000.0
Max NAV,10000000.0
Min NAV,10000000.0


In [8]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000) 
pd.set_option('display.max_rows', None)

# NOT `df` -- that name holds the day's tick data, still needed if you re-run
# the simulation cell above.
stock_df = pd.DataFrame(stockInfo)
print(stock_df)


Empty DataFrame
Columns: []
Index: []


----

## **Run Several Days in a Row**

Runs the strategy chosen above over several trading days back to back, the way the real daily run does:
each day loads the portfolio left by the day before, so a position bought on one day can be sold on a later one.

Run the cells up to **Craft Your Strategy** first (they load the strategy). No visual mode here — it runs headless and prints one row per day.

In [9]:
#TODO: The days to run, oldest first
trading_days = ["2026-09-09", "2026-09-10", "2026-09-11", "2026-09-14",
                "2026-09-15", "2026-09-16", "2026-09-17"]
# True  = delete result/<team_name>/ first, so the run starts from a fresh 10,000,000
# False = carry on from whatever portfolio is already saved there
fresh_start = True

if fresh_start:
    shutil.rmtree(os.path.join("result", team_name), ignore_errors=True)

strategy_dir = os.path.dirname(strategy_module.__file__)
day_rows = []

for day in trading_days:
    day_df = pd.read_csv(f"./marketInfo/ticks/{day}.csv")
    day_df['TradeDateTime'] = pd.to_datetime(day_df['TradeDateTime'])
    day_ticks = {symbol: group.to_dict("records") for symbol, group in day_df.groupby('ShareCode')}
    trading_date = day_df['TradeDateTime'].dt.date.iloc[0]

    # A new tradeSim per day loads result/<team_name>/<team_name>_portfolio.json,
    # which is what carries cash and holdings over from the previous day.
    trading_Sim = TradeSim.tradeSim(team_name, current_date=trading_date)
    strategy_runner = trading_Sim.get_strategy_runner()
    handlers = {
        symbol: StrategyHandler.StrategyHandler(strategy_class, strategy_runner, strategy_dir)
        for symbol in day_ticks
    }

    for tick in range(max(len(rows) for rows in day_ticks.values())):
        for symbol, rows in day_ticks.items():
            if tick >= len(rows):
                continue
            row = rows[tick]
            handlers[symbol].process_row(row)
            if not trading_Sim.isOrderbooksEmpty():
                trading_Sim.isMatch(row)
            trading_Sim.update_market_prices({row['ShareCode']: row['LastPrice']})

    # Same end-of-day order as the simulation cell above (see the note there)
    trading_Sim.flushTransactionLog()
    trading_Sim.flushErrorLogger()
    trading_Sim.create_transaction_summarize(team_name)
    _snapshots = trading_Sim.portfolio.daily_snapshots
    benchmark_start = _snapshots[0]["date"] if _snapshots else str(trading_date)
    trading_Sim.save_summary_csv(trading_date, university_start_date=benchmark_start)
    trading_Sim.save_portfolio()

    day_rows.append({
        "Date": day,
        "Cash": round(strategy_runner.get_cash_balance(), 2),
        "Return (%)": round(strategy_runner.get_roi(), 4),
        "Benchmark Return (%)": round(strategy_runner.get_benchmark_return(), 4),
        "Excess Return (%)": round(strategy_runner.get_excess_return(), 4),
        "Buy Orders": strategy_runner.get_buy_order_count(),
        "Sell Orders": strategy_runner.get_sell_order_count(),
        "Round-Trips": strategy_runner.get_round_trip_count(),
        "Holding": ", ".join(f"{s['Symbol']} {s['Actual Volume']}"
                             for s in strategy_runner.get_all_stocks_info()) or "-",
    })

# Buy/Sell/Round-Trip counts are running totals since the portfolio started
pd.DataFrame(day_rows)

[INFO] Created new portfolio for 'Full_option'
[OK] Summary saved to c:\Users\porsche\Documents\tradeSim0.2.5\result\Full_option\Full_option_portfolios_transaction_summary.csv
[INFO] Loaded existing portfolio from 'result\Full_option\Full_option_portfolio.json'
[OK] Summary saved to c:\Users\porsche\Documents\tradeSim0.2.5\result\Full_option\Full_option_portfolios_transaction_summary.csv
[INFO] Loaded existing portfolio from 'result\Full_option\Full_option_portfolio.json'
[OK] Summary saved to c:\Users\porsche\Documents\tradeSim0.2.5\result\Full_option\Full_option_portfolios_transaction_summary.csv
[INFO] Loaded existing portfolio from 'result\Full_option\Full_option_portfolio.json'
[OK] Summary saved to c:\Users\porsche\Documents\tradeSim0.2.5\result\Full_option\Full_option_portfolios_transaction_summary.csv
[INFO] Loaded existing portfolio from 'result\Full_option\Full_option_portfolio.json'
[OK] Summary saved to c:\Users\porsche\Documents\tradeSim0.2.5\result\Full_option\Full_option

,Date,Cash,Return (%),Benchmark Return (%),Excess Return (%),Buy Orders,Sell Orders,Round-Trips,Holding
0,2026-09-09,10000000.00,0.0000,0.0000,0.0000,0,0,0,-
1,2026-09-10,10000000.00,0.0000,-0.1163,0.1163,0,0,0,-
2,2026-09-11,8992760.78,-0.0764,-0.7820,0.7057,1,0,0,PTT 23800
3,2026-09-14,8012216.32,-0.0928,-1.1717,1.0788,2,0,0,"PTT 23800, KBANK 3900"
4,2026-09-15,8977791.52,-0.1071,-1.2557,1.1486,2,1,1,PTT 23800
5,2026-09-16,8977791.52,-0.2261,-2.3729,2.1468,2,1,1,PTT 23800
6,2026-09-17,8977791.52,-0.1666,-1.2271,1.0605,2,1,1,PTT 23800
